<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/10_text_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Analysis

In [1]:
import nltk

In [51]:
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [47]:
import polars as pl
from nltk.corpus import stopwords
from nltk.stem import (
    PorterStemmer,
    LancasterStemmer,
    RegexpStemmer,
    SnowballStemmer,
    WordNetLemmatizer as wnl,
)
from nltk.tag import pos_tag

In [ ]:
df = pl.read_csv("/content/global-cart.csv")
df

In [10]:
for idx, comment in enumerate(df['FeedbackText']):
  print(f"{idx+1}. {comment}\n")

1. The Pro-Grade Blender is a beast! It's powerful and quiet. Delivery was also incredibly fast, arrived in one day.

2. My package with the Comfort-Fit Running Shoes arrived damaged, and the box was completely crushed. Disappointed.

3. I had to return the blender. It was much larger than expected and didn't fit on my counter. The return was easy.

4. Are the Comfort-Fit shoes waterproof? Need to know before I buy. Your support chat is offline.

5. Fast shipping is great, but the Pro-Grade Blender's lid doesn't seem to seal properly. Seems like a defect.



In [15]:
feedback = df.select(pl.col('FeedbackText')).to_series().to_list()

tokens = nltk.word_tokenize(feedback[0])

print(tokens)

['The', 'Pro-Grade', 'Blender', 'is', 'a', 'beast', '!', 'It', "'s", 'powerful', 'and', 'quiet', '.', 'Delivery', 'was', 'also', 'incredibly', 'fast', ',', 'arrived', 'in', 'one', 'day', '.']


In [30]:
stop_words = set(stopwords.words('english'))
porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer('english')
regexp = RegexpStemmer('ing$|s$|e$|able$', min=4)
wordnet_lemmatizer = WordNetLemmatizer()

In [26]:
porter.stem("Hello")

'hello'

In [31]:
tokens_to_process = ['package', 'comfort-fit', 'running', 'shoes', 'arrived', 'damaged']
filtered_tokens = pl.DataFrame(
    {
        "tokens": [
            word for word in tokens_to_process
            if word.lower() not in stop_words
            ]
    }
    )
filtered_tokens = filtered_tokens.with_columns(
    pl.col("tokens").map_elements(
      porter.stem, return_dtype=pl.String).alias("porter_stemmed"),
    pl.col("tokens").map_elements(
      lancaster.stem, return_dtype=pl.String).alias("lancaster_stemmed"),
    pl.col("tokens").map_elements(
      snowball.stem, return_dtype=pl.String).alias("snowball_stemmed"),
)

display(filtered_tokens)

tokens,porter_stemmed,lancaster_stemmed,snowball_stemmed
str,str,str,str
"""package""","""packag""","""pack""","""packag"""
"""comfort-fit""","""comfort-fit""","""comfort-fit""","""comfort-fit"""
"""running""","""run""","""run""","""run"""
"""shoes""","""shoe""","""sho""","""shoe"""
"""arrived""","""arriv""","""ar""","""arriv"""
"""damaged""","""damag""","""dam""","""damag"""


In [65]:
tokens_to_process = nltk.word_tokenize("They refuse to permit us to obtain the refuse permit")
pos_tag(text)
# Penn Treebank
tagged_tokens = pl.DataFrame(
    pos_tag(tokens_to_process, tagset='universal'),
    schema=["token", "universal_pos_tag"],
    orient="row").join(
    pl.DataFrame(
        pos_tag(tokens_to_process),
        schema=["token", "penntree_tag"],
        orient="row"), on="token")
tagged_tokens

token,universal_pos_tag,penntree_tag
str,str,str
"""They""","""PRON""","""PRP"""
"""refuse""","""VERB""","""VBP"""
"""refuse""","""NOUN""","""VBP"""
"""to""","""PRT""","""TO"""
"""to""","""PRT""","""TO"""
…,…,…
"""the""","""DET""","""DT"""
"""refuse""","""VERB""","""NN"""
"""refuse""","""NOUN""","""NN"""


In [61]:
# [tt[1] for tt in pos_tag(tokens_to_process)]
pos_tag(tagged_tokens['token'])

[('They', 'PRP'),
 ('refuse', 'VBP'),
 ('to', 'TO'),
 ('permit', 'VB'),
 ('us', 'PRP'),
 ('to', 'TO'),
 ('obtain', 'VB'),
 ('the', 'DT'),
 ('refuse', 'NN'),
 ('permit', 'NN')]

In [49]:
# Valid POS options:
#    “n” for nouns,
#     “v” for verbs,
#     “a” for adjectives,
#     “r” for adverbs and
#     “s” for satellite adjectives.
wnl().lemmatize("package", "n")

'package'

In [ ]:


# Let's use our filtered tokens from before
tokens_to_process = ['package', 'comfort-fit', 'running', 'shoes', 'arrived', 'damaged']

# Why: pos_tag analyzes the list of words and assigns a part-of-speech tag to each one.
# For example, NN for singular noun, VBD for past tense verb.
tagged_tokens = pos_tag(tokens_to_process)